In [1]:
import pandas as pd

df = pd.read_csv("diamonds_cleaned.csv")
print("Original shape:", df.shape)

Original shape: (49851, 10)


In [2]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [3]:
features = df.drop(columns=["price"])
features.head()

,carat,depth,table,x,y,z,cut_encoded,color_encoded,clarity_encoded
0,0.23,61.5,55.0,3.95,3.98,2.43,5,6,2
1,0.21,59.8,61.0,3.89,3.84,2.31,4,6,3
2,0.23,56.9,65.0,4.05,4.07,2.31,2,6,5
3,0.29,62.4,58.0,4.20,4.23,2.63,4,2,4
4,0.31,63.3,58.0,4.34,4.35,2.75,2,1,2


In [4]:
features_with_const = add_constant(features)
features_with_const.head()

,const,carat,depth,table,x,y,z,cut_encoded,color_encoded,clarity_encoded
0,1.0,0.23,61.5,55.0,3.95,3.98,2.43,5,6,2
1,1.0,0.21,59.8,61.0,3.89,3.84,2.31,4,6,3
2,1.0,0.23,56.9,65.0,4.05,4.07,2.31,2,6,5
3,1.0,0.29,62.4,58.0,4.20,4.23,2.63,4,2,4
4,1.0,0.31,63.3,58.0,4.34,4.35,2.75,2,1,2


In [5]:
vif_data = pd.DataFrame()
vif_data["feature"] = features_with_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(features_with_const.values, i)
    for i in range(features_with_const.shape[1])
]

In [6]:
vif_data = vif_data[vif_data["feature"] != "const"]
vif_data = vif_data.sort_values(by="VIF", ascending=False)
vif_data

,feature,VIF
6,z,884.853312
5,y,643.666523
4,x,596.147891
1,carat,25.097771
2,depth,14.721902
3,table,1.626273
7,cut_encoded,1.505608
9,clarity_encoded,1.236523
8,color_encoded,1.120640


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [8]:
df = pd.read_csv("diamonds_cleaned.csv")
print(df.shape)
df.head()

(49851, 10)


,carat,depth,table,price,x,y,z,cut_encoded,color_encoded,clarity_encoded
0,0.23,61.5,55.0,326,3.95,3.98,2.43,5,6,2
1,0.21,59.8,61.0,326,3.89,3.84,2.31,4,6,3
2,0.23,56.9,65.0,327,4.05,4.07,2.31,2,6,5
3,0.29,62.4,58.0,334,4.20,4.23,2.63,4,2,4
4,0.31,63.3,58.0,335,4.34,4.35,2.75,2,1,2


In [9]:
X = df.drop(columns=["price"])
y = df["price"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (39880, 9)
Test shape: (9971, 9)


In [11]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train_log)

linear_pred_log = linear_model.predict(X_test_scaled)
linear_pred = np.expm1(linear_pred_log)

In [14]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train_log)

ridge_pred_log = ridge_model.predict(X_test_scaled)
ridge_pred = np.expm1(ridge_pred_log)

In [15]:
lasso_model = Lasso(alpha=0.001, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train_log)

lasso_pred_log = lasso_model.predict(X_test_scaled)
lasso_pred = np.expm1(lasso_pred_log)

In [16]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [17]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")
    return {"Model": model_name, "RMSE": rmse, "MAE": mae, "R2": r2}

In [18]:
results = []
results.append(evaluate(y_test, linear_pred, "Linear Regression"))
results.append(evaluate(y_test, ridge_pred, "Ridge Regression"))
results.append(evaluate(y_test, lasso_pred, "Lasso Regression"))
results.append(evaluate(y_test, rf_pred, "Random Forest"))

Linear Regression -> RMSE: 855.53, MAE: 454.89, R2: 0.9530
Ridge Regression -> RMSE: 855.58, MAE: 454.85, R2: 0.9530
Lasso Regression -> RMSE: 875.83, MAE: 459.65, R2: 0.9507
Random Forest -> RMSE: 529.04, MAE: 263.48, R2: 0.9820


In [19]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="R2", ascending=False)
results_df

,Model,RMSE,MAE,R2
3,Random Forest,529.041733,263.484262,0.982017
0,Linear Regression,855.533701,454.885512,0.952972
1,Ridge Regression,855.580830,454.848418,0.952966
2,Lasso Regression,875.831868,459.651601,0.950713


In [20]:
from sklearn.linear_model import RidgeCV

alphas_to_try = [0.01, 0.1, 1, 10, 50, 100, 200, 500]
ridge_cv = RidgeCV(alphas=alphas_to_try, cv=5)
ridge_cv.fit(X_train_scaled, y_train_log)

print("Best alpha:", ridge_cv.alpha_)

Best alpha: 10.0


In [30]:
from sklearn.linear_model import LassoCV

lasso_cv = LassoCV(alphas=[0.0001, 0.0005, 0.001, 0.005, 0.01, 0.02, 0.05], cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train_log)

print("Best alpha:", lasso_cv.alpha_)


Best alpha: 0.0001


In [40]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model(model, X_test_scaled, y_test):
    y_pred_log = model.predict(X_test_scaled)
    y_pred = np.expm1(y_pred_log)  # reverse log1p

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"R²:   {r2:.4f}")

    return rmse, mae, r2

print("Ridge:")
ridge_metrics = evaluate_model(ridge_cv, X_test_scaled, y_test)

print("\nLasso:")
lasso_metrics = evaluate_model(lasso_cv, X_test_scaled, y_test)

Ridge:
RMSE: 856.36
MAE:  454.69
R²:   0.9529

Lasso:
RMSE: 855.68
MAE:  454.49
R²:   0.9530


In [31]:
X_reduced = X.drop(columns=["x", "y", "z"])
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

y_train_r_log = np.log1p(y_train_r)

reduced_model = LinearRegression()
reduced_model.fit(X_train_r_scaled, y_train_r_log)

print("Carat coefficient (no x/y/z in model):", reduced_model.coef_[0])

Carat coefficient (no x/y/z in model): 1.0309557106944787


In [32]:
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importances = importances.sort_values(by="Importance", ascending=False)
importances

,Feature,Importance
0,carat,0.485414
4,y,0.401791
8,clarity_encoded,0.062783
7,color_encoded,0.030894
5,z,0.006217
3,x,0.005925
1,depth,0.003025
2,table,0.002289
6,cut_encoded,0.001663


In [33]:
from sklearn.linear_model import ElasticNet, ElasticNetCV

In [34]:
elastic_model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000)
elastic_model.fit(X_train_scaled, y_train_log)

elastic_pred_log = elastic_model.predict(X_test_scaled)
elastic_pred = np.expm1(elastic_pred_log)

In [35]:
evaluate(y_test, elastic_pred, "Elastic Net (baseline)")

Elastic Net (baseline) -> RMSE: 867.65, MAE: 457.16, R2: 0.9516


{'Model': 'Elastic Net (baseline)',
 'RMSE': np.float64(867.6548150166882),
 'MAE': 457.1585087291633,
 'R2': 0.9516295012507314}

In [36]:
l1_ratios_to_try = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]

elastic_cv = ElasticNetCV(
    l1_ratio=l1_ratios_to_try,
    alphas=[0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    cv=5,
    max_iter=10000
)
elastic_cv.fit(X_train_scaled, y_train_log)

print("Best alpha:", elastic_cv.alpha_)
print("Best l1_ratio:", elastic_cv.l1_ratio_)

Best alpha: 0.0005
Best l1_ratio: 0.1


In [37]:
elastic_cv_pred = np.expm1(elastic_cv.predict(X_test_scaled))
evaluate(y_test, elastic_cv_pred, "Elastic Net (CV-tuned)")

Elastic Net (CV-tuned) -> RMSE: 858.31, MAE: 454.96, R2: 0.9527


{'Model': 'Elastic Net (CV-tuned)',
 'RMSE': np.float64(858.3081134337375),
 'MAE': 454.9646860455328,
 'R2': 0.9526660182839719}

In [41]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [42]:
df = pd.read_csv("diamonds_cleaned.csv")
print(df.shape)

(49851, 10)


In [43]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (39880, 9)
Test shape: (9971, 9)


In [44]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")
    return {"Model": model_name, "RMSE": rmse, "MAE": mae, "R2": r2}

In [45]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

In [46]:
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=10,        # was 20
    cv=2,              # was 3
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

In [47]:
rf_search.fit(X_train, y_train)
print("Best parameters:", rf_search.best_params_)

Best parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 20}


In [48]:
rf_best = rf_search.best_estimator_
rf_best_pred = rf_best.predict(X_test)

evaluate(y_test, rf_best_pred, "Random Forest (tuned)")

Random Forest (tuned) -> RMSE: 527.50, MAE: 262.69, R2: 0.9821


{'Model': 'Random Forest (tuned)',
 'RMSE': np.float64(527.4982270899454),
 'MAE': 262.69440365676024,
 'R2': 0.9821216015669281}

In [50]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np

def evaluate_model(model, X_test_feat, y_test, log_target=True):
    y_pred = model.predict(X_test_feat)
    if log_target:
        y_pred = np.expm1(y_pred)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return rmse, mae, r2

results = {}

results["Linear Regression"]        = evaluate_model(linear_model, X_test_scaled, y_test, log_target=True)
results["Ridge (CV Tuned)"]         = evaluate_model(ridge_cv, X_test_scaled, y_test, log_target=True)
results["Lasso (CV Tuned)"]         = evaluate_model(lasso_cv, X_test_scaled, y_test, log_target=True)
results["Elastic Net (CV Tuned)"]   = evaluate_model(elastic_cv, X_test_scaled, y_test, log_target=True)
results["Random Forest (CV Tuned)"] = evaluate_model(rf_best, X_test, y_test, log_target=False)

results_df = pd.DataFrame(results, index=["RMSE", "MAE", "R2"]).T
results_df = results_df.sort_values("R2", ascending=False)
results_df

,RMSE,MAE,R2
Random Forest (CV Tuned),527.498227,262.694404,0.982122
Linear Regression,855.533701,454.885512,0.952972
Lasso (CV Tuned),855.677032,454.494963,0.952956
Ridge (CV Tuned),856.357621,454.690088,0.952881
Elastic Net (CV Tuned),858.308113,454.964686,0.952666


In [51]:
%store linear_model
%store ridge_cv
%store lasso_cv
%store elastic_cv
%store rf_best
%store X_test_scaled
%store X_test
%store y_test

Stored 'linear_model' (LinearRegression)
Stored 'ridge_cv' (RidgeCV)
Stored 'lasso_cv' (LassoCV)
Stored 'elastic_cv' (ElasticNetCV)
Stored 'rf_best' (RandomForestRegressor)
Stored 'X_test_scaled' (ndarray)
Stored 'X_test' (DataFrame)
Stored 'y_test' (Series)
